## This is the notebook to generate eigenanchors for other datasets


In [1]:
# Imports

import numpy as np
from scipy.interpolate import splprep, splev
import os
import json
from scipy.linalg import svd
import glob
import os
from tools.data.textdet.totaltext_converter import get_contours_mat

def resample_polygon(polygon, num_points):
    """
    Resample a polygon to have a specified number of points.
    Args:
        polygon (list of tuples): List of (x, y) coordinates defining the polygon.
        num_points (int): Desired number of points in the resampled polygon.
    Returns:
        np.ndarray: Resampled polygon as a numpy array of shape (num_points, 2).
    """

    # Ensure the polygon is a numpy array and closed
    polygon = np.array(polygon, dtype=np.float32)
    if not np.allclose(polygon[0], polygon[-1]):
        polygon = np.vstack((polygon, polygon[0]))
    
    # extract coordinates
    x, y = polygon[:, 0], polygon[:, 1]
    # Create a parameterization of the polygon
    # In other words, make a curve from the polygon points
    # s=0 means no smoothing, per=True means the curve is periodic (closed)
    tck, _ = splprep([x, y], s=0, per=True)
    
    u_new = np.linspace(0, 1, num_points+1)[:-1]  # Exclude the last point to avoid duplication
    x_new, y_new = splev(u_new, tck) # Get new x and y coordinates
    return np.ravel(np.stack([x_new, y_new], axis=1))

def generate_eigenanchors(polygons, num_points=14):
    """
    Build a contour matrix from a list of polygons.
    Args:
        polygons (list of list of tuples): List of polygons, each defined by a list of [x, y] coordinates.
        num_points (int): Number of points to resample each polygon to.
    Returns:
        np.ndarray: Contour matrix with shape (num_polygons, num_points, 2).
    """
    print(f"Resampling polygons to {num_points} points each.")
    # Resample each polygon to have the same number of points
    vectors = []
    for p in polygons:
        if len(p) < 4:
            continue
        try:
            resampled_polygon = resample_polygon(p, num_points)
            vectors.append(resampled_polygon)
        except Exception as e:
            print(f"Skipping invalid polygon: {e}")

    A = np.stack(vectors, axis=1)

    # Use SVD according to the paper
    U, S, Vt = svd(A, full_matrices=False)
    eigenanchors = U[:, :14] # Take the first 14 eigenanchors
    return eigenanchors.T


RuntimeError: KeyboardInterrupt: 

In [ ]:
# Generate eigenanchors from a list of polygons make 40
# Example polygons (list of polygons, each defined by a list of (x, y) tuples)
polygons = [
    [(0, 0), (1, 0), (1, 1), (0, 1)],
    [(2, 2), (3, 2), (3, 3), (2, 3)],
    [(4, 4), (5, 4), (5, 5), (4, 5)],
    [(6, 6), (7, 6), (7, 7), (6, 7)],
    [(8, 8), (9, 8), (9, 9), (8, 9)],
    [(10, 10), (11, 10), (11, 11), (10, 11)],
    [(12, 12), (13, 12), (13, 13), (12, 13)],
    [(14, 14), (15, 14), (15, 15), (14, 15)],
    [(16, 16), (17, 16), (17, 17), (16, 17)],
    [(18, 18), (19, 18), (19, 19), (18, 19)],
    [(20, 20), (21, 20), (21, 21), (20, 21)],
    [(22, 22), (23, 22), (23, 23), (22, 23)],
    [(24, 24), (25, 24), (25, 25), (24, 25)],
    [(26, 26), (27, 26), (27, 27), (26, 27)],
    [(28, 28), (29, 28), (29, 29), (28, 29)],
    [(30, 30), (31, 30), (31, 31), (30, 31)],
    [(32, 32), (33, 32), (33, 33), (32, 33)],
    [(34, 34), (35, 34), (35, 35), (34, 35)],
    [(36, 36), (37, 36), (37, 37), (36, 37)],
    [(38, 38), (39, 38), (39, 39), (38, 39)],
    [(40, 40), (41, 40), (41, 41), (40, 41)],
    [(42, 42), (43, 42), (43, 43), (42, 43)],
    [(44, 44), (45, 44), (45, 45), (44, 45)]]
eigenanchors = generate_eigenanchors(polygons, num_points=14)
print("Eigenanchors shape:", eigenanchors.shape)
print("Eigenanchors:", eigenanchors[:, 0])


In [2]:
gt_dir = "../data/CTW1500/instances_training.json"  # or "test"

with open(gt_dir, "r") as f:
    coco = json.load(f)

all_polygons = []

for ann in coco["annotations"]:
    for seg in ann["segmentation"]:
        coords = np.array(seg, dtype=np.float32).reshape(-1, 2)
        if coords.shape[0] >= 3:  # Only valid polygons
            all_polygons.append(coords)

eigenanchors = generate_eigenanchors(all_polygons, num_points=14)
eigenanchors = eigenanchors.astype(np.float32)  # <--- Add this line
print("Eigenanchors shape:", eigenanchors.shape)

# Save the eigenanchors to a file
np.savez("ctw1500_eigenanchors.npz", components_c=eigenanchors)



Resampling polygons to 14 points each.
Eigenanchors shape: (14, 28)


In [ ]:
eigenanchors

In [ ]:
import numpy as np
A = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12], [13, 14], [15, 16]])
print("Original")
print(A)

print("Transposed")
print(A.transpose())


Original
[[ 1  2]
 [ 3  4]
 [ 5  6]
 [ 7  8]
 [ 9 10]
 [11 12]
 [13 14]
 [15 16]]
Transposed
[[ 1  3  5  7  9 11 13 15]
 [ 2  4  6  8 10 12 14 16]]
